# Preparación de datos con Python: estructura y filtros

## Breve descripción de la sesión

Limpieza y preparación de la información en su formato y forma según objetivos del diseño, asegurando la relevancia del insumo previo a análisis y procesos. Junto con la sesión anterior, es uno de los procesos más importantes y con alta asignación de esfuerzo. Se debe tener claro el objetivo de diseño para poder definir estructuras y filtros que permitan que los datos sean relevantes previo a cualquier análisis o proceso. Errores en la sesión 2 o 3 generarán un efecto dominó hacia cualquiera de las sesiones posteriores, mientras más lejos, más grave y más costoso será corregir.

## Repaso de la sesión 2

* Acceso a la documentación.
* Lectura de archivos.
* Estadística descriptiva básica: `shape()`, `describe()`, `info()`.

In [ ]:
# Librerías cargadas
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', 500)

## Acceso a la documentación

Documentación oficial:
- [Documentación oficial de python](https://python.readthedocs.io/en/latest/tutorial/index.html) 
- [Documentación oficial de numpy](https://docs.scipy.org/doc/numpy/user/quickstart.html) 
- [Documentación oficial de pandas](http://pandas.pydata.org/pandas-docs/stable/)

Para búsquedas usando el entorno de desarrollo:
- En jupyter, usar el operador `?` luego del nombre de un método
- El método `help()`, nativo de python.

Documentación/solución de problemas informal:
- Google, buscando la pregunta que desean solucionar: Ej: "sort dataframe column ascending pandas python". De los resultados, se recomienda revisar aquellos provenientes del portal `stackOverflow`.

In [ ]:
list?

In [ ]:
help(list)

## Lectura de archivos

In [ ]:
### Lectura de archivos
# Pandas provee read_csv, que lee un archivo generando un dataFrame

headers = ['imdbID', 'title', 'year', 'score', 'votes', 'runtime', 'genres'] 
movies = pd.read_csv("data/imdb_top_10000.txt", sep="\t", header=None, names=headers, encoding='UTF-8')
movies.head()

 ## Estadística descriptiva básica

In [ ]:
movies.shape # tamaño del dataset en (filas, columnas)

In [ ]:
movies.describe() # 

In [ ]:
movies.info() # info sobre cantidad de filas, datatypes y tamaño ocupado en memoria

* Evitar en lo posible objetos de tipo `object`, pues reducen el potencial de pandas. En datasets muy grandes esto se vuelve crítico.

El archivo parece estar en un buen estado para trabajarlo.

- *¿Entiendo las columnas y los datos que tengo?*

Teniendo fijado el objetivo de lo que se quiere plantear:

- *¿Están los datos que necesito para reponder las preguntas/solucionar el objetivo planteado?*
- *¿Me sirven las columnas tal como están o tengo que arreglar los datos?*

**Ambas preguntas son difíciles de responder, pero anticiparse a ello puede evitar la pérdida de tiempo, habiendo avanzado en la solución del objetivo. **

Antes de trabajar, respaldaremos los datos ante cualquier problema. Tendremos la versión original y a la que se le harán cambios.

In [ ]:
# IMPORTANTE: respaldar el dataframe original previo a trabajar en él en una variable auxiliar.
# Cuando el código está correcto y funcional, se puede remover esta linea y sus implicancias de forma segura.

movies_raw = movies.copy()

Utilizando el mismo ejercicio de la clase anterior, profundizaremos en los conceptos de filtro y estructuras de los datos.

In [ ]:
movies.head()

Tal como se pudo ver en la tarea de la sesión anterior, el archivo cargado tiene ciertos detalles:

- Columnas `idIMDB` y `runtime`,  son del tipo `object` -> convertirlas en algo más manipulable puede ayudarnos a futuro.
- Remover la infomación del año de la columna `title` -> info duplicada si se considera la columna `year`.
- El género está como un string -> transformar a un formato fácil de filtrar. 

Observamos que cada fila del campo `imdbID` tiene un valor de la forma `tt` + número.

In [ ]:
# Se concatenan métodos
# método 1: considerar el string desde la posición 2 en adelante
# método 2: convertir a int usando numpy

movies["imdbID"] = movies["imdbID"].str[2:].astype(np.int)
movies.head()

La columna `runtime` tiene un valor de la forma número + ` mins.`.

In [ ]:
# quitar los strings de la columna runtime para tener solo los valores enteros

movies["runtime"] = movies["runtime"].str.split(' ').str.get(0).astype(np.int)
movies.head()

Fíjese que si se quiere buscar la película más corta, aparece una de duración especial...

In [ ]:
# Película cuyo índice tenga el menor valor del campo 'runtime'
movies.loc[movies["runtime"].idxmin(), :]

Ante lo anterior, se convertirán a `NaN` todos aquellos valores en que la duración de la película sea igual a cero.

In [ ]:
# Quitar aquellas películas con duración igual a cero, posibles errores
movies["runtime"] = movies["runtime"].replace(0, np.nan)

In [ ]:
# Película cuyo índice tenga el menor valor del campo 'runtime'
movies.loc[movies["runtime"].idxmin(), :]

## Ejercicio en clase \#1 - Repaso 

Remover el valor del año desde la columna `title` del dataFrame e imprimir el nombre de las 5 películas mejor calificadas.

Ejemplo: `The Shawshank Redemption (1994)` debe pasar a `The Shawshank Redemption`

In [ ]:
movies_ej1 = movies.copy()

In [ ]:
movies_ej1.head()

In [ ]:
# Ordenadas por 'score'

movies_ej1 = movies_ej1.sort_values('score', ascending=False)
movies_ej1.head()

In [ ]:
# Ahora filtrando el año del campo title.
# Forma fácil, pero riesgosa en caso de que no todos los datos estén así de limpios: ' (XXXX)'

title_clean1 = movies_ej1["title"].str[:-7].copy()
title_clean1.head()

In [ ]:
# Una forma más segura para dividir strings es usando la combinación split-get
title_clean2 = movies_ej1["title"].str.split(' \(').str.get(0)
title_clean2.head()

In [ ]:
# Usando el segundo método

movies_ej1["title"] = movies_ej1["title"].str.split(' \(').str.get(0)
movies_ej1.head()

In [ ]:
# Aplicamos el cambio al dataframe original para continuar con él así
movies = movies_ej1.copy()

# Fin de ejercicio

# Sesión 3: preparación de datos con Python - Filtros

La manera más fácil de entender un filtro en un conjunto de datos, independiente del lenguaje de programación, es que **sirve para resolver preguntas que aparezcan sobre los datos**. Suelen generar un subconjunto del mismo, facilitando la obtención de la respuesta a la pregunta.

Hay diferentes formas de plantearlos: 
- Filtro usando los métodos `loc` e `iloc`.
- Filtros booleano con operadores aritméticos.
- Métodos propios para filtrar: isnull(), isin().

## Filtro usando los métodos loc e iloc

Son muy potentes pues se pueden usar rangos para filtrar. Recuerde lo ya visto:

- loc: explícito (índices_filas,nombres_columnas)
- iloc: implícito (posicion_filas, posicion_columnas)

In [ ]:
# Filtro usando loc
#Se seleccionan los datos del 0 al 1 en el índice, independiente de la posición y el orden, y las columnas 'title', 'year' y 'score'.

movies.loc[0:8, ['title', 'year', 'score']].sort_values('score', ascending=False) # índice + nombres_columnas

In [ ]:
# Filtro usando iloc

#Con el método iloc no importan cuales sean los valores del índice, este selecciona por la posición. 
#Por ejemplo para seleccionar los datos de la posición 2 a 5 y las columnas de las posiciones 1, 2 y 3:

movies.iloc[2:6, [1,2,3]] # Posiciones

## Filtro booleano con operadores aritméticos

El objetivo es aplicar condiciones sobre las columnas del dataframe que generen expresiones booleanas (True or False) a través de operaciones aritméticas.

Una pregunta simple: *¿Qué películas se estrenaron después de 1980?*

In [ ]:
movies_year_exp = movies['year'] > 1980
movies_year_exp.head(11)

Cuando se evalúa una operación dentro de un dataframe, lo hace a lo largo de cada fila, implicando que permanezcan solo aquellas filas que cumplan la condición. Compare los índices entre ambos casos para verificar lo anterior.

In [ ]:
movies_year_df1 = movies[movies['year'] > 1980]
movies_year_df1.head()

A no confiarse, porque las preguntas no suelen ser siempre así de fáciles...

*¿Qué películas se estrenaron después de 1980, cuyo género contenga el drama y tenga una puntuación mayor o igual a 8.0 para más de 350000 votantes?*

In [ ]:
movies_year_df2 = movies[(movies['year'] > 1980) & (movies['genres'].str.contains('Drama')) & (movies['score'] >= 8.0) & (movies['votes'] > 350000)]
movies_year_df2.head(3)

Note que, finalmente, la muestra se redujo de 10000 a 7 registros tras aplicar los filtros.

## Métodos propios para filtrar: isnull( )

Cuando hay `NaN`s dentro del conjunto de datos, se pueden identificar con este método.

In [ ]:
# Encontrar NaN en la columna genres
movies_genres_null = movies[movies['genres'].isnull()]
movies_genres_null

Más adelante operaremos sobre el valor `NaN` de este registro.

## Métodos propios para filtrar: isin( )

El método `isin()` permite saber si una lista, diccionario, serie y/o dataframe están dentro del dataframe a filtrar.

Repasando los datos ya vistos:

In [ ]:
movies.head()

### isin( ) - lista como parámetro 

In [ ]:
# Lista como parámetro de ingreso: se revisa cada elemento en cada fila y se determina si se cumple o no.
movies['title'].isin(['The Godfather','Pulp Fiction', 'Deadpool']).head()

In [ ]:
# Lista como parámetro de ingreso
movies_isin_list = movies[movies['title'].isin(['The Godfather','Pulp Fiction', 'Deadpool'])].head()
movies_isin_list

### isin( ) - diccionario como parámetro 

In [ ]:
# Diccionario como parámetro de ingreso
movies.isin({'title':['The Godfather']}).head()

In [ ]:
# Completamos toda la información necesaria.
movies_isin_dicc = movies[movies.isin({'title':['The Godfather','Pulp Fiction'],
                                     'imdbID':[110912,68646],
                                     'year':[1994,1972],
                                     'votes':[474189,490065],
                                     'score':[9.0,9.2],
                                     'genres':['Crime|Drama','Crime|Thriller'],
                                     'runtime':[154,175],
                                    })].head()
movies_isin_dicc

Al ingresar un diccionario como atributo evaluador, aparecen valores con `NaN`. 
- Eliminar aquellas filas que contengan una falta de coincidencia (False/NaN).

In [ ]:
movies_isin_dicc.dropna()

### isin( ) - Series / Dataframe como parámetro 

In [ ]:
# Series/Dataframe como parámetro de ingreso
df_aux = pd.DataFrame({'title':['The Godfather','Pulp Fiction'],
                                     'imdbID':[110912,68646],
                                     'year':[1994,1972],
                                     'votes':[474189,490065],
                                     'score':[9.0,9.2],
                                     'genres':['Crime|Drama','Crime|Thriller'],
                                     'runtime':[154,175] })
df_aux

In [ ]:
movies_isin_df = movies[movies.isin(df_aux)]
movies_isin_df.head()

Note que la principal diferencia entre pasar el diccionario y el dataframe como parámetro es que para el primero no importa el orden de apariencia. 

- Si el valor en par clave-valor del diccionario coincide en cualquier posición con el de una columna con el mismo nombre dataframe original, entonces se obtiene True. 
- Al pasar el nuevo dataframe generado, solo se obtiene True si la ubicación es exactamente la misma al original.

## Ejercicio en clase \#2 - Filtros

Usando el mismo conjunto de datos, responda usando el dataframe movies_ej2:

In [ ]:
movies_ej2 = movies.copy()

a. ¿Cuántas películas de 'El Padrino' (The Godfather) están dentro de las mejores 10000 de IMDB?

Genere un dataframe con la siguiente información: nombre de la película, año de estreno, calificación, duración, opinión breve de cada uno de los integrantes acerca de la película.

In [ ]:
# Usando isin() solo se obtienen coincidencias exactas.
movies_godfather = movies_ej2[movies_ej2['title'].isin(['The Godfather'])]
movies_godfather

In [ ]:
# Usando str.contains se pueden ver todas las coincidencias
movies_godfather = movies_ej2[movies_ej2['title'].str.contains('The Godfather')]
movies_godfather

In [ ]:
# Dejando el dataframe tal como se requiere
movies_godfather = movies_godfather[['title','year','score','runtime']].copy()
movies_godfather.loc[movies_godfather.year == 1972,'commentary FPB'] = 'Peliculaza! Marlon Brando acariciando al gato de imprevisto y fuera de libreto...'
movies_godfather.loc[movies_godfather.year == 1974,'commentary FPB'] = 'Al nivel de la 1!'
movies_godfather.loc[movies_godfather.year == 1990,'commentary FPB'] = 'Menos relevante que las dos anteriores...'

movies_godfather

b. Si le piden recomendaciones de películas que sean de drama y crimen, estrenadas en la década del '90 y considerando el 4% de las mejores calificadas, ¿cuáles serían tres de sus recomendaciones?¿Coincide con la calificación expuesta?

- Se espera un dataframe que respete el máximo de filas enunciado anteriormente (aproxime de ser necesario).
- Que este tenga ÚNICAMENTE la información elemental de la película (nombre, año de estreno, calificación, duración y género en forma de palabra).
- Agregue dos columnas adicionales: una en la que califique usted a la película con nota de 1 a 10 y otra en la que indique su recomendación.

In [ ]:
# Revisamos los datos actuales
movies_ej2.head()

In [ ]:
# Se genera el dataframe
movies_ej2_b = movies_ej2[((movies_ej2["genres"].str.contains("Drama")) & (movies_ej2["genres"].str.contains("Crime"))) & (movies_ej2["year"]>=1990) & (movies_ej2["year"]<=1999)]
movies_ej2_b = movies_ej2_b.sort_values('score', ascending=False).copy()
movies_ej2_b.head()

In [ ]:
# % de las películas seleccionadas
movies_count = int(round(len(movies_ej2_b.index)*0.04))
movies_count

In [ ]:
# Se corta el df asociado, dejando solo el top de filas
# Recuerde que el .copy() es para evitar cambios en
#movies_ej2_b_top = movies_ej2_b.iloc[:movies_count , np.r_[1:4, 5:7]].copy()
movies_ej2_b_top = movies_ej2_b.iloc[:movies_count , np.r_[1:4, 5:7]].copy()
movies_ej2_b_top.head()

# Considerar columnas literales

In [ ]:
# Manera limpia y fácil de conseguir el filtro esperado

movies_ej2_b_top = movies_ej2_b[['title','year','score','runtime','genres']].sort_values('score', ascending=False).head(movies_count).copy()
movies_ej2_b_top.head()

In [ ]:
# Se agregan 2 columnas
# Se agrega numeración
movies_ej2_b_top.loc[:22,'score FPB'] = 9.4
movies_ej2_b_top.loc[22:,'score FPB'] = 8.5
# Se agrega la recomendación
movies_ej2_b_top.loc[21,'must to watch by FPB'] = 'Yes!'
movies_ej2_b_top.loc[0,'must to watch by FPB'] = 'Yes!'
movies_ej2_b_top.loc[17,'must to watch by FPB'] = 'Yes!'
movies_ej2_b_top = movies_ej2_b_top.fillna('')
movies_ej2_b_top.head(3)

In [ ]:
len(movies_ej2_b_top.index) # Largo igual al % de filas requerido

# Sesión 3: preparación de datos con Python - Estructura

Una forma de entener la importancia de esta sección es considerando el cambio de estructura que puede generar la resolución de una pregunta.

El tercer cambio mencionado en el repaso de la sesión anterior es el de añadir la columna `genres` al dataframe en formato *one_hot_encoding* (Tarea 2 - Ejercicio 3). Previo a eso, en la sección anterior adelantamos que se aprecia que esta columna pareciera tener un valor nulo.

In [ ]:
movies.info()

In [ ]:
# Ver campo nulo en género

movies[movies["genres"].isnull()]

In [ ]:
# Usamos .loc para llegar al índice de la fila que cumpla la condición 
# y seleccionar la columna que queremos cambiar.

movies.loc[movies["imdbID"] == 990404,"genres"] = 'Drama'
movies.loc[movies["imdbID"] == 990404,]

In [ ]:
movies.info()

**Considere lo anterior como algo que ocurre con gran frecuencia durante el proceso de aquisición y limpieza de datos.**
- Es muy difícil que estos queden tal cual como se requieren los datos por diferentes razones que obligan a poner en pausa el trabajo.
- Errores y el reajuste de los objetivos sobre la marcha son algunos de los factores principales.

Teniendo los datos limpios, se puede ver qué ocurre con el *one-hot encoding*:

In [ ]:
one_hot_encoding = movies["genres"].str.get_dummies(sep='|') # guardamos el encoding en un nuevo dataframe
one_hot_encoding.head()

Removemos la columna `genres` que tenía los campos strings:

In [ ]:
movies = movies.drop(columns=['genres'])

Haciendo la unión por índice entre ambos dataframes:

In [ ]:
movies = pd.concat([movies, one_hot_encoding], axis=1)
movies.head(3)

## Ejercicio en clase \#3 - Estructura

Para el siguiente ejercicio, responda lo siguiente usando el dataframe movies_ej3:

In [ ]:
movies_ej3 = movies.copy()

a. Si quiero ver las 5 mejores películas de ciencia ficción (Sci-Fi), ¿cuántas horas se necesitan?

In [ ]:
# Filtrar dataframe por las condiciones esperadas.

movies_ej3_a = movies_ej3[movies_ej3['Sci-Fi'] == 1].sort_values('score', ascending=False).head(5).copy()
movies_ej3_a

In [ ]:
# Sumar los tiempos y convertirlos a horas.

movies_ej3_a_min = movies_ej3_a['runtime'].sum()/60 
sol_ej3_a = 'Se requieren '+ str(movies_ej3_a_min) +' horas para ver las mejores cinco películas de ciencia ficción.'
sol_ej3_a

b. Conocido es el top 250 de IMDB, que presenta las películas mejores calificadas según usuarios calificadores de todo el mundo. Se requiere determinar los 3 géneros mejores calificados, junto con la duración promedio de las películas y el promedio de votantes de los 3 peores calificados para este subconjunto de películas.

In [ ]:
# Ordenamos por 'score' y filtramos las 250 mejores películas
movies_ej3_b = movies_ej3.sort_values('score', ascending=False).iloc[:250,:]
movies_ej3_b.head()

# Agregamos una fila adicional
#movies_ej3_b.loc['Suma'] = pd.Series(movies_ej3_b['Action'].sum(), index = ['Action'])
#movies_ej3_b



## ¡Con esto termina la sesión 3!